## downlading the libraies and local model

In [1]:
# 1️⃣ تثبيت المكتبات المطلوبة
!pip install -q ollama pydantic pandas tabulate nest_asyncio

print("✅ تم تثبيت المكتبات بنجاح")

✅ تم تثبيت المكتبات بنجاح


In [2]:
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
import time

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

!ollama --version

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cli.github.com/packages stable InRelease [4,685 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [113 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,620 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/multiverse amd64 Packages [93.2 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,623 kB

In [3]:
!ollama pull qwen2.5:14b

In [4]:
# 2️⃣ الاستيراد وإعداد البيئة المحلية
import asyncio
import json
import os
from concurrent.futures import ThreadPoolExecutor
from enum import Enum
from typing import List, Optional
import nest_asyncio
from ollama import chat
from pydantic import BaseModel, Field, field_validator

# تفعيل nest_asyncio لتشغيل asyncio داخل بيئات Jupyter/Colab
nest_asyncio.apply()

# مسمى النموذج المحلي (Qwen 2.5 14B أو DeepSeek-R1 14B)
LOCAL_MODEL_NAME = "qwen2.5:14b"

# التحقق من تشغيل Ollama محلياً
try:
    response = chat(
        model=LOCAL_MODEL_NAME,
        messages=[{"role": "user", "content": "ping"}],
    )
    print(f"✅ الاتصال بنجاح مع النموذج المحلي: {LOCAL_MODEL_NAME}")
except Exception as e:
    print(
        f"⚠️ تنبيه: يرجى التأكد من تشغيل Ollama محلياً وتنفيذ الأمر (ollama pull {LOCAL_MODEL_NAME}).\nالخطأ: {e}"
    )

✅ الاتصال بنجاح مع النموذج المحلي: qwen2.5:14b


In [5]:
# 3️⃣ بناء هيكل البيانات المنسق (Pydantic Schemas with CoT & Legal Citations)


class RiskLevel(str, Enum):
    RED = "أحمر"  # خطر مرتفع / بطلان مطلق / شرط تعسفي
    YELLOW = "أصفر"  # خطر متوسط / غموض / شرط انحيازي
    GREEN = "أخضر"  # آمن / متوافق مع أحكام القانون


class ClauseRiskAnalysis(BaseModel):
    clause_id: int = Field(description="رقم البند")
    clause_label: str = Field(description="عنوان أو مسمى البند")
    clause_text: str = Field(description="النص الأصلي للبند")

    reasoning_steps: List[str] = Field(
        description="خطوات التحليل المنطقي والقانوني خطوة بخطوة (Chain of Thought)"
    )
    risk_level: RiskLevel = Field(
        description="مستوى الخطورة: أحمر (مرتفع)، أصفر (متوسط)، أخضر (آمن)"
    )
    risk_score: int = Field(
        description="درجة الخطورة من 1 (آمن جداً) إلى 10 (باطل ومجحف للغاية)"
    )

    is_void_legal_term: bool = Field(
        description="هل البند يعتبر باطلاً بطلاناً مطلقاً أو نسبياً وفقاً للقانون المصري؟"
    )
    cited_law_articles: List[str] = Field(
        description="أسماء وأرقام المواد القانونية المعتمد عليها المأخوذة من الـ RAG"
    )

    simple_explanation: str = Field(
        description="شرح مبسط للبند ولماذا يشكل خطورة بلغة عامية بسيطة"
    )
    legal_rationale: str = Field(
        description="التعليل القانوني الدقيق والربط بالمواد القانونية المطبقة"
    )
    suggested_balanced_clause: Optional[str] = Field(
        default=None,
        description="الصياغة البديلة المقترحة والمعدلة لتحقيق التوازن العقدية",
    )

    @field_validator("risk_score")
    def check_score_range(cls, v: int) -> int:
        if not 1 <= v <= 10:
            raise ValueError("Risk score must be between 1 and 10")
        return v


class OverallSummarySchema(BaseModel):
    contract_title: str = Field(
        description="مسمى العقد بناءً على تحليل المخرجات"
    )
    overall_risk_score: int = Field(
        description="التقييم الإجمالي للعقد من 1 إلى 10"
    )
    overall_risk_summary: str = Field(
        description="ملخص تنفيذي للمخاطر وتوصيات قانونية شامله للمستخدم"
    )


print("✅ تم تعريف هياكل البيانات للتحليل القانوني بنجاح")

✅ تم تعريف هياكل البيانات للتحليل القانوني بنجاح


In [6]:
# 4️⃣ دمج وقراءة مخرجات ملفات الـ OCR (Task 1) والـ RAG (Task 3)


def load_ocr_output(file_path: str = "ocr_output.json") -> tuple:
    """قراءة مخرجات ملف الـ OCR وتقسيم البنود"""
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            print(f"📄 تم تحميل ملف الـ OCR بنجاح: {file_path}")
            return data.get("metadata", {}), data.get("clauses", [])

    print("⚠️ لم يتم العثور على ocr_output.json، سيتم استخدام بيانات نموذجية.")
    mock_metadata = {
        "contract_type": "عقد إيجار مسكن",
        "parties": "المؤجر: أحمد محمود | المستأجر: مينا سامي",
    }
    mock_clauses = [
        {
            "clause_id": 1,
            "clause_label": "البند الأول - مدة العقد والقيمة الإيجارية",
            "clause_text": "مدة العقد سنة واحدة تبدأ من 1/1/2026 وتجدد تلقائياً، والإيجار 5000 جنيه شهرياً.",
        },
        {
            "clause_id": 2,
            "clause_label": "البند الثاني - الإخلاء والشرط الجزائي",
            "clause_text": "إذا تأخر المستأجر يومين عن الدفع، يحق للمؤجر طرده فوراً وتغيير الكوالين دون حكم قضائي مع شرط جزائي 100,000 جنيه.",
        },
        {
            "clause_id": 3,
            "clause_label": "البند الثالث - الإعفاء من الضمان والصيانة",
            "clause_text": "يتنازل المستأجر عن حق مطالبة المؤجر بأي صيانة للعين أو ضمان العيوب الخفية مهما كانت جسامتها.",
        },
    ]
    return mock_metadata, mock_clauses


def load_rag_context(file_path: str = "rag_output.json") -> str:
    """قراءة مخرجات استرجاع الـ RAG للمواد القانونية"""
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            print(f"📚 تم تحميل ملف الـ RAG بنجاح: {file_path}")
            if isinstance(data, list):
                return "\n".join(
                    [f"- {item.get('text', item)}" for item in data]
                )
            elif isinstance(data, dict):
                return data.get("context_text", str(data))

    print("⚠️ لم يتم العثور على rag_output.json، سيتم استخدام مواد نموذجية.")
    return """
- المادة (571) من القانون المدني المصري: يلتزم المؤجر بصيانة العين المؤجرة لتبقى صالحة للاستيفاء بالنفع المقصود.
- المادة (577) من القانون المدني المصري: يضمن المؤجر للمستأجر العيوب التي تحول دون الانتفاع، ويكون باطلاً كل اتفاق يعفي المؤجر من ضمان العيوب إذا تعمد إخفاءها.
- المادة (224) من القانون المدني المصري: يجوز للقاضي تخفيض الشرط الجزائي المبالغ فيه.
- قواعد آمرة: لا يجوز الإخلاء القسري للعين المؤجرة دون حكم قضائي واجب النفاذ.
"""


contract_metadata, contract_clauses = load_ocr_output()
legal_rag_context = load_rag_context()

⚠️ لم يتم العثور على ocr_output.json، سيتم استخدام بيانات نموذجية.
⚠️ لم يتم العثور على rag_output.json، سيتم استخدام مواد نموذجية.


In [7]:
# 5️⃣ بناء المحرك المحلي للتحليل المعزز (System Prompt + CoT + Parallel Execution)

SYSTEM_PROMPT = """
أنت مستشار قانوني خبير بالقانون المدني وقوانين العقود المصرية.
مهمتك تحليل البند المقدم لك مقارنة بالمواد القانونية المرفقة من الـ RAG.

التزم بالتعليمات التالية:
1. قم بالتحليل على خطوات متسلسلة (Chain of Thought) في الميدان `reasoning_steps`.
2. استخرج أرقام واسماء المواد القانونية المستند إليها وضعها في `cited_law_articles`.
3. حدد درجة الخطورة من 1 إلى 10:
   - (1-3): بند آمن (أخضر)
   - (4-6): بند ينطوي على انحياز أو غموض (أصفر)
   - (7-10): بند باطل أو تعسفي يخالف القواعد الآمرة (أحمر)
4. قدم صياغة بديلة متوازنة إذا كانت درجة الخطورة أعلى من 3.
"""


def process_single_clause_with_retry(
    clause: dict, metadata: dict, rag_context: str, retries: int = 2
) -> ClauseRiskAnalysis:
    """معالجة بند واحد عبر Ollama المحلي مع إعادة المحاولة عند الخطأ"""
    user_prompt = f"""
📌 **بيانات العقد:** {metadata.get('contract_type', 'عقد')}
📖 **المواد القانونية المتاحة (RAG Context):**
{rag_context}

📝 **البند المطلوب تحليله:**
- الرقم: {clause.get('clause_id')}
- العنوان: {clause.get('clause_label')}
- النص: {clause.get('clause_text')}
"""
    for attempt in range(retries + 1):
        try:
            response = chat(
                model=LOCAL_MODEL_NAME,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                format=ClauseRiskAnalysis.model_json_schema(),
                options={"temperature": 0.1},
            )
            return ClauseRiskAnalysis.model_validate_json(
                response.message.content
            )
        except Exception as e:
            if attempt == retries:
                raise RuntimeError(
                    f"فشل تحليل البند {clause.get('clause_id')}: {e}"
                )


async def run_parallel_analysis(
    metadata: dict, clauses: List[dict], rag_context: str
) -> dict:
    """تشغيل التحليل بالتوازي للبنية التحتية المحلية"""
    loop = asyncio.get_event_loop()
    print(f"⚡ جاري تحليل {len(clauses)} بنود بالتوازي عبر النموذج المحلي...")

    with ThreadPoolExecutor(max_workers=3) as executor:
        tasks = [
            loop.run_in_executor(
                executor,
                process_single_clause_with_retry,
                clause,
                metadata,
                rag_context,
            )
            for clause in clauses
        ]
        results: List[ClauseRiskAnalysis] = await asyncio.gather(*tasks)

    # توليد الملخص الإجمالي للعقد
    summary_prompt = f"""
بناءً على نتائج البنود المحللة التالية:
{json.dumps([c.model_dump() for c in results], ensure_ascii=False)}

قم بتوليد الملخص التنفيذي وتقييم العقد الإجمالي.
"""
    summary_res = chat(
        model=LOCAL_MODEL_NAME,
        messages=[{"role": "user", "content": summary_prompt}],
        format=OverallSummarySchema.model_json_schema(),
        options={"temperature": 0.2},
    )
    summary_obj = OverallSummarySchema.model_validate_json(
        summary_res.message.content
    )

    return {"summary": summary_obj, "clauses_analysis": results}


print("✅ تم إعداد محرك المعالجة المتوازية بنجاح")

✅ تم إعداد محرك المعالجة المتوازية بنجاح


In [8]:
# 6️⃣ تنفيذ التحليل الشامل للعقد
full_report_data = asyncio.run(
    run_parallel_analysis(
        contract_metadata, contract_clauses, legal_rag_context
    )
)

print("\n🎉 اكتمل التحليل القانوني المحلي بنجاح!")
print(f"📋 العقد: {full_report_data['summary'].contract_title}")
print(
    f"🔢 التقييم الإجمالي للمخاطر: {full_report_data['summary'].overall_risk_score}/10"
)

⚡ جاري تحليل 3 بنود بالتوازي عبر النموذج المحلي...


KeyboardInterrupt: 

In [ ]:
# 7️⃣ عرض تقرير المخاطر التفاعلي (HTML Dashboard Viewer)
from IPython.display import HTML, display


def display_rich_dashboard(data: dict):
    summary: OverallSummarySchema = data["summary"]
    clauses: List[ClauseRiskAnalysis] = data["clauses_analysis"]

    html = f"""
    <div style="font-family: 'Segoe UI', Tahoma, sans-serif; direction: rtl; text-align: right; line-height: 1.6; max-width: 950px; margin: auto;">
        <div style="background: linear-gradient(135deg, #0f172a, #1e293b); color: white; padding: 25px; border-radius: 12px; margin-bottom: 25px; box-shadow: 0 4px 6px rgba(0,0,0,0.2);">
            <div style="display: flex; justify-content: space-between; align-items: center;">
                <h2 style="margin: 0; color: #38bdf8;">⚖️ تقرير تحليل مخاطر العقد: {summary.contract_title}</h2>
                <span style="background-color: #ef4444; color: white; font-size: 1.1em; padding: 6px 16px; border-radius: 20px; font-weight: bold;">
                    درجة الخطورة: {summary.overall_risk_score}/10
                </span>
            </div>
            <p style="margin-top: 15px; color: #cbd5e1; font-size: 1.05em;"><b>الملخص التنفيذي:</b> {summary.overall_risk_summary}</p>
        </div>
    """

    for item in clauses:
        color_map = {
            RiskLevel.RED: (
                "#ef4444",
                "#fef2f2",
                "#fca5a5",
                "🔴 خطر مرتفع / شرط باطل",
            ),
            RiskLevel.YELLOW: (
                "#f59e0b",
                "#fffbeb",
                "#fde68a",
                "🟡 تحذير / غير متوازن",
            ),
            RiskLevel.GREEN: (
                "#10b981",
                "#ecfdf5",
                "#6ee7b7",
                "🟢 متوافق / آمن",
            ),
        }
        badge_color, bg_color, border_color, risk_title = color_map[
            item.risk_level
        ]

        steps_html = "".join(
            [f"<li>{step}</li>" for step in item.reasoning_steps]
        )
        laws_html = ", ".join(item.cited_law_articles)

        html += f"""
        <div style="background-color: {bg_color}; border: 1.5px solid {border_color}; border-radius: 10px; padding: 18px; margin-bottom: 20px;">
            <div style="display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid {border_color}; padding-bottom: 10px; margin-bottom: 12px;">
                <h3 style="margin: 0; color: #0f172a;">📌 {item.clause_label}</h3>
                <div>
                    <span style="background-color: #334155; color: white; padding: 4px 10px; border-radius: 12px; font-size: 0.85em; margin-left: 5px;">
                        الخطورة: {item.risk_score}/10
                    </span>
                    <span style="background-color: {badge_color}; color: white; padding: 4px 12px; border-radius: 12px; font-weight: bold; font-size: 0.85em;">
                        {risk_title}
                    </span>
                </div>
            </div>

            <p><b>📜 النص الأصلي:</b> <i style="color: #334155;">"{item.clause_text}"</i></p>

            <div style="background-color: rgba(255,255,255,0.7); padding: 10px; border-radius: 6px; margin: 10px 0;">
                <b>🧠 خطوات التفكير المنطقي (CoT):</b>
                <ol style="margin: 5px 0 0 20px; padding: 0; color: #334155;">{steps_html}</ol>
            </div>

            <p><b>💡 الشرح المبسط:</b> {item.simple_explanation}</p>
            <p><b>⚖️ التعليل القانوني:</b> {item.legal_rationale}</p>
            <p style="color: #1e40af;"><b>📚 المواد الاسترشادية:</b> <span style="background-color: #dbeafe; padding: 2px 8px; border-radius: 4px;">{laws_html}</span></p>

            {'<p style="color: #dc2626; font-weight: bold;">⚠️ تنبيه: هذا البند يخالف القواعد القانونية الآمرة ويعتبر باطلاً.</p>' if item.is_void_legal_term else ''}

            {f'''<div style="background-color: #ffffff; padding: 12px; border-right: 4px solid #10b981; margin-top: 10px; border-radius: 4px;">
                <b>🛠️ الصياغة البديلة المقترحة:</b><br>{item.suggested_balanced_clause}
            </div>''' if item.suggested_balanced_clause else ''}
        </div>
        """

    html += "</div>"
    display(HTML(html))


display_rich_dashboard(full_report_data)

In [ ]:
# 8️⃣ تصدير النتائج النهائية إلى ملفات JSON و Pandas DataFrame
import pandas as pd

# 1. تحويل البنود لـ DataFrame
df_clauses = pd.DataFrame(
    [c.model_dump() for c in full_report_data["clauses_analysis"]]
)

# 2. حفظ التقرير الشامل
output_filename = "contract_risk_analysis_output.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(
        {
            "summary": full_report_data["summary"].model_dump(),
            "clauses_analysis": [
                c.model_dump() for c in full_report_data["clauses_analysis"]
            ],
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(f"✅ تم تصدير نتائج التقييم النهائي بنجاح إلى الملف: {output_filename}")
display(
    df_clauses[
        [
            "clause_id",
            "clause_label",
            "risk_score",
            "risk_level",
            "is_void_legal_term",
        ]
    ]
)

### constructing a new contract based on user input


In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional
from enum import Enum
from ollama import chat
import json

# 1. مدخلات المستخدم لإصدار العقد
class ContractUserInput(BaseModel):
    contract_type: str = Field(description="نوع العقد (مثال: عقد إيجار مسكن، عقد عمل، عقد بيع)")
    party_one_name: str = Field(description="اسم الطرف الأول وصفته (مثال: المؤجر / البائع)")
    party_two_name: str = Field(description="اسم الطرف الثاني وصفته (مثال: المستأجر / المشتري)")
    key_terms: List[str] = Field(description="الشروط الأساسية (القيمة المالية، المدة، مكان العين، إلخ)")
    special_requests: Optional[List[str]] = Field(default=[], description="أي شروط خاصة يطلبها المستخدم")

# 2. هيكل العقد المولد قانونياً
class ContractClauseDraft(BaseModel):
    clause_number: int = Field(description="رقم البند")
    clause_title: str = Field(description="عنوان البند")
    clause_text: str = Field(description="النص القانوني الملزم للبند")
    legal_basis: str = Field(description="المادة القانونية المعتمد عليها من القانون المصري لضمان شرعية البند")
    balance_explanation: str = Field(description="توضيح كيف يحقق هذا البند التوازن وعدم الإجحاف بأحد الطرفين")

class FullContractDocument(BaseModel):
    contract_title: str = Field(description="العنوان الرسمي للعقد")
    preamble: str = Field(description="ديباجة العقد وتحديد هوية الأطراف والأهلية القانونية")
    clauses: List[ContractClauseDraft] = Field(description="قائمة بنود العقد المصاغة قانونياً")
    closing_and_signatures: str = Field(description="صيغة الخاتمة والإقرار بالاستلام وعدد النسخ والتوقيعات")

In [ ]:

SYSTEM_DRAFTING_PROMPT = """
أنت محامي وصائغ عقود خبير في القانون المدني وقوانين التجارة والعمل المصرية.
مهمتك كتابة عقد قانوني مكتمل الأركان ومتوازن بناءً على مدخلات المستخدم والمواد القانونية المرفقة من الـ RAG.

التزم بالقواعد التالية أثناء الصياغة:
1. **التوازن العقدي:** عدم تضمين أي شروط تعسفية أو باطلة بطلاناً مطلقاً وفقاً للقانون المصري.
2. **الاستناد إلى القانون:** ربط كل بند بالمادة القانونية المناسبة المأخوذة من الـ RAG Context.
3. **الدقة والصياغة الرسمية:** استخدام مصطلحات قانونية مصاغة بوضوح يمنع التأويل أو الغموض.
4. **حماية الطرفين:** التأكد من إدراج حقوق وواجبات متكافئة للطرفين.
"""

def generate_balanced_contract(
    user_input: ContractUserInput,
    rag_legal_context: str,
    model_name: str = "qwen2.5:7b"
) -> FullContractDocument:
    """توليد عقد متكامل بناءً على مدخلات المستخدم والمواد القانونية"""

    prompt = f"""
📌 **مدخلات المستخدم لإنشاء العقد:**
- نوع العقد: {user_input.contract_type}
- الطرف الأول: {user_input.party_one_name}
- الطرف الثاني: {user_input.party_two_name}
- الشروط الأساسية: {json.dumps(user_input.key_terms, ensure_ascii=False)}
- طلبات خاصة: {json.dumps(user_input.special_requests, ensure_ascii=False)}

📖 **المواد القانونية المعتمدة المسترجعة من الـ RAG:**
{rag_legal_context}

قم بصياغة العقد كاملاً واستخراج النتيجة بالصيغة الهيكلية المطلوبة.
"""

    response = chat(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM_DRAFTING_PROMPT},
            {"role": "user", "content": prompt}
        ],
        format=FullContractDocument.model_json_schema(),
        options={"temperature": 0.2}
    )

    return FullContractDocument.model_validate_json(response.message.content)

In [ ]:
from IPython.display import HTML, display

def render_generated_contract_html(contract: FullContractDocument):
    clauses_html = ""
    for clause in contract.clauses:
        clauses_html += f"""
        <div style="margin-bottom: 20px; border-bottom: 1px dashed #cbd5e1; padding-bottom: 15px;">
            <h4 style="color: #1e293b; margin-bottom: 5px;">📌 {clause.clause_title} (بند {clause.clause_number})</h4>
            <p style="font-size: 1.1em; color: #0f172a; background: #f8fafc; padding: 10px; border-right: 3px solid #0284c7; border-radius: 4px;">
                {clause.clause_text}
            </p>
            <div style="font-size: 0.88em; color: #475569; display: flex; gap: 15px;">
                <span><b>⚖️ السند القانوني:</b> {clause.legal_basis}</span>
                <span><b>🛡️ التوازن العقدي:</b> {clause.balance_explanation}</span>
            </div>
        </div>
        """

    html = f"""
    <div style="font-family: 'Amiri', 'Traditional Arabic', 'Segoe UI', serif; direction: rtl; text-align: right; line-height: 1.8; max-width: 850px; margin: auto; padding: 30px; border: 2px solid #0f172a; border-radius: 8px; background: #ffffff; box-shadow: 0 10px 15px rgba(0,0,0,0.05);">
        <h2 style="text-align: center; color: #0f172a; border-bottom: 2px solid #0f172a; padding-bottom: 10px; margin-bottom: 20px;">
            📜 {contract.contract_title}
        </h2>

        <div style="background-color: #f1f5f9; padding: 15px; border-radius: 6px; margin-bottom: 25px;">
            <h4 style="margin: 0 0 8px 0; color: #334155;">📖 الديباجة والأطراف:</h4>
            <p style="margin: 0; color: #1e293b;">{contract.preamble}</p>
        </div>

        <h3 style="color: #0f172a; border-bottom: 1px solid #e2e8f0; padding-bottom: 5px;">بنود العقد</h3>
        {clauses_html}

        <div style="background-color: #f8fafc; padding: 15px; border: 1px solid #e2e8f0; border-radius: 6px; margin-top: 30px;">
            <h4 style="margin: 0 0 8px 0; color: #334155;">✍️ الإقرار والتوقيعات:</h4>
            <p style="margin: 0 0 15px 0; color: #1e293b;">{contract.closing_and_signatures}</p>
            <div style="display: flex; justify-content: space-between; margin-top: 25px; padding: 0 20px;">
                <div style="text-align: center;"><b>الطرف الأول:</b><br><br>____________________</div>
                <div style="text-align: center;"><b>الطرف الثاني:</b><br><br>____________________</div>
            </div>
        </div>
    </div>
    """
    display(HTML(html))

In [ ]:
# 1. إدخال بيانات طلب العقد من المستخدم
user_req = ContractUserInput(
    contract_type="عقد إيجار شقة سكنية",
    party_one_name="محمود السيد علي (مؤجر)",
    party_two_name="أحمد حسن محمد (مستأجر)",
    key_terms=[
        "العين المؤجرة: شقة رقم 4 بالدور الثالث بالمعادي",
        "المدة: سنة واحدة تبدأ من 1-10-2026",
        "القيمة الإيجارية: 6000 جنيه مصري شهرياً تُدفع في بداية كل شهر"
    ],
    special_requests=[
        "إلزام المؤجر بإجراء الصيانة الأساسية والعمومية",
        "تحديد شرط جزائي متوازن يوازي إيجار شهر واحد فقط في حال الإخلال"
    ]
)

# 2. المواد القانونية المأخوذة من الـ RAG لتوجيه الصياغة
rag_context = """
- المادة (558) مدني مصري: الإيجار عقد يلتزم المؤجر بمقتضاه أن مكن المستأجر من الانتفاع بعين معينة لمدة محددة لقاء أجر معلوم.
- المادة (571) مدني مصري: يلتزم المؤجر بصيانة العين المؤجرة لتبقى على الحالة التي صالحة معها للاستيفاء بالنفع المقصود.
- المادة (224) مدني مصري: يجوز للقاضي تخفيض الشرط الجزائي إذا كان مبالغاً فيه.
"""

# 3. تشغيل الصياغة وعرض العقد
generated_contract = generate_balanced_contract(user_req, rag_context, model_name="qwen2.5:7b")
render_generated_contract_html(generated_contract)